# Подготовка целевого признака ухода клиента

В этом ноутбуке мы определяем, кого считать ушедшим клиентом, и собираем обучающую выборку для модели прогноза ухода.

## Выбранная логика

По данным видно, что очень большая часть клиентов делает только один заказ. Если строить модель на всех клиентах сразу, она будет почти всегда предсказывать уход, и такая постановка будет слабой с точки зрения бизнеса.

Поэтому берём более осмысленную задачу:

- рассматриваем клиентов, у которых уже был хотя бы второй заказ или есть признак лояльности;
- на некоторую дату-срез смотрим их историю;
- если в следующие `180` дней клиент не совершил ни одного заказа, считаем его ушедшим.

Такой подход лучше подходит для удержания ценных клиентов и хорошо объясняется на защите.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)

BASE_DIR = Path("..")
DATA_PATH = BASE_DIR / "data.csv"
EVENTS_PATH = BASE_DIR / "events.csv"


In [2]:
def parse_dt(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, utc=True, errors="coerce", format="mixed")


def build_orders_table(data_path: Path) -> pd.DataFrame:
    cols = ["order_id", "order_item_id", "user_id", "created_at", "sale_price", "status", "is_loyal"]
    data = pd.read_csv(data_path, usecols=cols)
    data["created_at"] = parse_dt(data["created_at"])

    orders = (
        data.groupby("order_id", as_index=False)
        .agg(
            user_id=("user_id", "first"),
            created_at=("created_at", "min"),
            order_value=("sale_price", "sum"),
            items_count=("order_item_id", "nunique"),
            status=("status", "first"),
            is_loyal=("is_loyal", "max"),
        )
        .dropna(subset=["user_id", "created_at"])
        .sort_values(["user_id", "created_at"])
    )
    orders["created_at_naive"] = orders["created_at"].dt.tz_localize(None)
    return orders


def build_events_table(events_path: Path) -> pd.DataFrame:
    cols = ["id", "user_id", "session_id", "sequence_number", "created_at", "traffic_source", "browser", "uri", "event_type"]
    events = pd.read_csv(events_path, usecols=cols)
    events["created_at"] = parse_dt(events["created_at"])
    events = events.dropna(subset=["user_id", "created_at"]).copy()
    events["user_id"] = events["user_id"].astype(int)
    events["created_at_naive"] = events["created_at"].dt.tz_localize(None)
    return events


In [3]:
orders = build_orders_table(DATA_PATH)
events = build_events_table(EVENTS_PATH)

orders.shape, events.shape


((125085, 8), (2616836, 10))

## 1. Выбираем дату-срез

Для примера возьмём дату `2025-06-01`.

- всё, что было до неё, используем как историю клиента;
- следующие `180` дней используем для разметки ухода.

Позже можно сделать несколько таких срезов и собрать более сильную обучающую выборку.

In [4]:
CUTOFF_DATE = pd.Timestamp("2025-06-01")
LOOKBACK_DAYS = 365
CHURN_HORIZON_DAYS = 180

LOOKBACK_START = CUTOFF_DATE - pd.Timedelta(days=LOOKBACK_DAYS)
HORIZON_END = CUTOFF_DATE + pd.Timedelta(days=CHURN_HORIZON_DAYS)

LOOKBACK_START, CUTOFF_DATE, HORIZON_END


(Timestamp('2024-06-01 00:00:00'),
 Timestamp('2025-06-01 00:00:00'),
 Timestamp('2025-11-28 00:00:00'))

## 2. Определяем, кого включаем в задачу

Берём клиентов, которые уже достаточно важны для бизнеса:

- сделали хотя бы два заказа к дате-срезу;
- или помечены как лояльные.

In [5]:
history_orders = orders.loc[orders["created_at_naive"] < CUTOFF_DATE].copy()

user_history = (
    history_orders.groupby("user_id", as_index=False)
    .agg(
        total_orders_before_cutoff=("order_id", "nunique"),
        loyal_flag=("is_loyal", "max"),
        first_order_at=("created_at_naive", "min"),
        last_order_at=("created_at_naive", "max"),
    )
)

eligible_users = user_history.loc[
    (user_history["total_orders_before_cutoff"] >= 2) | (user_history["loyal_flag"] == True),
    "user_id"
]

len(eligible_users)


20886

## 3. Собираем признаки по истории клиента

Используем окно истории длиной `365` дней до даты-среза. Это даёт достаточно данных о поведении клиента, но не смешивает слишком старую активность с более свежей.

In [6]:
orders_lookback = history_orders.loc[
    (history_orders["created_at_naive"] >= LOOKBACK_START) &
    (history_orders["created_at_naive"] < CUTOFF_DATE)
].copy()

order_features = (
    orders_lookback.groupby("user_id", as_index=False)
    .agg(
        orders_last_365d=("order_id", "nunique"),
        revenue_last_365d=("order_value", "sum"),
        avg_order_value_last_365d=("order_value", "mean"),
        median_order_value_last_365d=("order_value", "median"),
        avg_items_last_365d=("items_count", "mean"),
        last_order_at=("created_at_naive", "max"),
        first_order_in_window_at=("created_at_naive", "min"),
    )
)

order_features["days_since_last_order"] = (CUTOFF_DATE - order_features["last_order_at"]).dt.days
order_features["active_days_in_window"] = (order_features["last_order_at"] - order_features["first_order_in_window_at"]).dt.days
order_features.head()


,user_id,orders_last_365d,revenue_last_365d,avg_order_value_last_365d,median_order_value_last_365d,avg_items_last_365d,last_order_at,first_order_in_window_at,days_since_last_order,active_days_in_window
0,1,1,166.470005,166.470005,166.470005,1.0,2024-10-14 10:31:40,2024-10-14 10:31:40,229,0
1,2,1,372.000000,372.000000,372.000000,2.0,2025-04-11 13:43:06,2025-04-11 13:43:06,50,0
2,16,1,203.969994,203.969994,203.969994,1.0,2025-01-15 01:23:56,2025-01-15 01:23:56,136,0
3,18,1,50.909998,50.909998,50.909998,1.0,2025-05-25 19:38:14,2025-05-25 19:38:14,6,0
4,24,2,304.500000,152.250000,152.250000,1.0,2025-02-27 23:52:23,2024-12-10 09:32:36,93,79


In [7]:
events_lookback = events.loc[
    (events["created_at_naive"] >= LOOKBACK_START) &
    (events["created_at_naive"] < CUTOFF_DATE)
].copy()

events_lookback["is_product"] = events_lookback["event_type"].eq("product").astype(int)
events_lookback["is_cart"] = events_lookback["event_type"].eq("cart").astype(int)
events_lookback["is_purchase"] = events_lookback["event_type"].eq("purchase").astype(int)
events_lookback["is_department"] = events_lookback["event_type"].eq("department").astype(int)

event_features = (
    events_lookback.groupby("user_id", as_index=False)
    .agg(
        sessions_last_365d=("session_id", "nunique"),
        events_last_365d=("id", "count"),
        product_views_last_365d=("is_product", "sum"),
        cart_events_last_365d=("is_cart", "sum"),
        purchase_events_last_365d=("is_purchase", "sum"),
        department_events_last_365d=("is_department", "sum"),
        last_event_at=("created_at_naive", "max"),
    )
)
event_features["days_since_last_event"] = (CUTOFF_DATE - event_features["last_event_at"]).dt.days
event_features["events_per_session_last_365d"] = event_features["events_last_365d"] / event_features["sessions_last_365d"]
event_features.head()


,user_id,sessions_last_365d,events_last_365d,product_views_last_365d,cart_events_last_365d,purchase_events_last_365d,department_events_last_365d,last_event_at,days_since_last_event,events_per_session_last_365d
0,1,1,10,2,2,2,2,2024-10-14 10:19:27,229,10.0
1,2,2,28,8,8,4,8,2025-04-14 11:54:55,47,14.0
2,16,1,10,2,2,2,2,2025-01-14 23:56:56,137,10.0
3,18,1,10,2,2,2,2,2025-05-25 19:38:03,6,10.0
4,24,2,20,4,4,4,4,2025-02-27 21:42:28,93,10.0


## 4. Строим целевой признак ухода

Если клиент не сделал ни одного заказа в следующие `180` дней после даты-среза, ставим `churn_target = 1`.

In [8]:
future_orders = orders.loc[
    (orders["created_at_naive"] >= CUTOFF_DATE) &
    (orders["created_at_naive"] < HORIZON_END)
].copy()

future_buyers = set(future_orders["user_id"].unique())

target = user_history[user_history["user_id"].isin(eligible_users)].copy()
target["churn_target"] = (~target["user_id"].isin(future_buyers)).astype(int)
target["future_purchase_flag"] = 1 - target["churn_target"]
target[["user_id", "total_orders_before_cutoff", "loyal_flag", "churn_target"]].head()


,user_id,total_orders_before_cutoff,loyal_flag,churn_target
1,2,2,True,1
5,8,1,True,0
6,10,1,True,1
7,13,2,False,1
8,14,1,True,0


In [9]:
model_dataset = (
    target
    .merge(order_features, on="user_id", how="left")
    .merge(event_features, on="user_id", how="left")
)

model_dataset = model_dataset.sort_values("user_id").reset_index(drop=True)
model_dataset.head()


,user_id,total_orders_before_cutoff,loyal_flag,first_order_at,last_order_at_x,churn_target,future_purchase_flag,orders_last_365d,revenue_last_365d,avg_order_value_last_365d,median_order_value_last_365d,avg_items_last_365d,last_order_at_y,first_order_in_window_at,days_since_last_order,active_days_in_window,sessions_last_365d,events_last_365d,product_views_last_365d,cart_events_last_365d,purchase_events_last_365d,department_events_last_365d,last_event_at,days_since_last_event,events_per_session_last_365d
0,2,2,True,2020-06-24 23:00:38,2025-04-11 13:43:06,1,0,1.0,372.0,372.0,372.0,2.0,2025-04-11 13:43:06,2025-04-11 13:43:06,50.0,0.0,2.0,28.0,8.0,8.0,4.0,8.0,2025-04-14 11:54:55,47.0,14.0
1,8,1,True,2023-11-27 12:56:00,2023-11-27 12:56:00,0,1,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN
2,10,1,True,2023-03-06 18:01:15,2023-03-06 18:01:15,1,0,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN
3,13,2,False,2022-08-12 17:01:29,2023-03-21 15:22:16,1,0,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN
4,14,1,True,2020-04-18 04:44:08,2020-04-18 04:44:08,0,1,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN


In [10]:
summary = pd.Series(
    {
        "eligible_users": len(model_dataset),
        "churn_share": round(model_dataset["churn_target"].mean(), 4),
        "avg_orders_before_cutoff": round(model_dataset["total_orders_before_cutoff"].mean(), 2),
        "share_loyal": round(model_dataset["loyal_flag"].mean(), 4),
    }
).to_frame("value")
summary


,value
eligible_users,20886.0000
churn_share,0.8061
avg_orders_before_cutoff,2.1400
share_loyal,0.5781


In [11]:
display(model_dataset.groupby("loyal_flag")["churn_target"].mean().rename("churn_share").to_frame())
display(
    model_dataset.groupby(pd.cut(model_dataset["total_orders_before_cutoff"], bins=[0, 1, 2, 4, 10], include_lowest=True))["churn_target"]
    .mean()
    .rename("churn_share")
    .to_frame()
)


,churn_share
loyal_flag,
False,0.960281
True,0.693639


/var/folders/2z/_grmn59568vdk2lt6w8h6p180000gn/T/ipykernel_33209/1131970170.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  model_dataset.groupby(pd.cut(model_dataset["total_orders_before_cutoff"], bins=[0, 1, 2, 4, 10], include_lowest=True))["churn_target"]


,churn_share
total_orders_before_cutoff,
"(-0.001, 1.0]",0.592462
"(1.0, 2.0]",0.870081
"(2.0, 4.0]",0.808393
"(4.0, 10.0]",NaN


## 5. Как это интерпретировать

Эта постановка не пытается предсказать судьбу всех клиентов подряд. Она отвечает на более полезный вопрос:

**какие уже важные для бизнеса клиенты могут перестать покупать в ближайшие 180 дней?**

Именно такую задачу удобнее связать с удержанием, рекомендациями и персональными действиями.

## Что дальше

Следующий шаг:

- выбрать признаки для первой модели;
- разделить данные на обучение и проверку;
- построить базовую модель ухода;
- оценить важность признаков и проверить интерпретируемость.